<a href="https://colab.research.google.com/github/Sevin-Tayfehaliverdi/bioinformatics-lab/blob/main/Bio5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install biopython -q

import random
from Bio import Entrez
import pandas as pd

Entrez.email = "student@example.com"

def fetch_ncbi_sequence(accession="NC_001422.1"):
    try:
        handle = Entrez.efetch(db="nucleotide", id=accession, rettype="fasta", retmode="text")
        fasta_record = handle.read()
        handle.close()

        lines = fasta_record.splitlines()
        seq = "".join(line for line in lines if not line.startswith(">"))
        return seq.upper()
    except Exception as e:
        print(f"NCBI fetch warning ({e}), using fallback synthetic DNA sequence.")
        bases = ['A', 'C', 'G', 'T']
        return "".join(random.choice(bases) for _ in range(2000))

dna_sequence = fetch_ncbi_sequence("NC_001422.1")
if len(dna_sequence) > 3000:
    dna_sequence = dna_sequence[:2500]
elif len(dna_sequence) < 1000:
    bases = ['A', 'C', 'G', 'T']
    dna_sequence += "".join(random.choice(bases) for _ in range(1500 - len(dna_sequence)))

print(f"Successfully obtained DNA sequence! Length: {len(dna_sequence)} nucleotides\n")

def brute_force_repeat_detector(seq, min_len=3, max_len=6, min_count=3):
    repeat_results = []

    for length in range(min_len, max_len + 1):
        substring_counts = {}
        substring_positions = {}

        for i in range(len(seq) - length + 1):
            subseq = seq[i : i + length]

            if not all(base in "ACGT" for base in subseq):
                continue

            substring_counts[subseq] = substring_counts.get(subseq, 0) + 1
            if subseq not in substring_positions:
                substring_positions[subseq] = []
            substring_positions[subseq].append(i)

        for subseq, count in substring_counts.items():
            if count >= min_count:
                repeat_results.append({
                    "Pattern": subseq,
                    "Length": length,
                    "Count": count,
                    "Positions": substring_positions[subseq]
                }
                )

    return pd.DataFrame(repeat_results)

print("Running Brute-Force DNA Repeat Detection (length 3 to 6)...")
results_df = brute_force_repeat_detector(dna_sequence, min_len=3, max_len=6, min_count=4)

if not results_df.empty:
    results_df = results_df.sort_values(by=["Count", "Length"], ascending=[False, False]).reset_index(drop=True)
    print(f"\nFound {len(results_df)} repeating patterns meeting the criteria:\n")
    print(results_df.head(15).to_string())
else:
    print("No recurring patterns found with the current threshold. Try lowering min_count.")


Successfully obtained DNA sequence! Length: 2500 nucleotides

Running Brute-Force DNA Repeat Detection (length 3 to 6)...

Found 611 repeating patterns meeting the criteria:

   Pattern  Length  Count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            Positions
0      TTT       3     92  [3, 4, 35, 44, 96, 177, 186, 276, 293, 294, 325, 326, 418, 430, 465, 466, 472, 487, 493, 494, 513, 564, 581, 599, 617, 695, 755, 935, 958, 959, 1043, 1091, 1154, 1199, 1200, 1201, 1206, 1207, 1208, 1229, 1323, 1324, 1368, 1372, 1383, 1404, 1405, 1479, 150